### Catastrophe Historical Aggregation Pipeline

Creates (or updates) a serverless Lakeflow Declarative Pipeline that rolls the
`{CATALOG}.simulator.catastrophe_hist_*` Delta tables (generated by the
`Catastrophe_History` stage) into a Bronze → Silver → Gold medallion in
`{CATALOG}.lakeflow`.

Runs the pipeline once (triggered) and waits for completion, so the gold
analytics tables exist by the time this stage finishes. Depends on
`Catastrophe_History` (the source tables must already exist).

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
SIMULATOR_SCHEMA = dbutils.widgets.get("SIMULATOR_SCHEMA")

# The historical source tables land in {CATALOG}.{SIMULATOR_SCHEMA} as
# catastrophe_hist_* (see the Catastrophe_History stage). Gold/silver outputs
# land in {CATALOG}.lakeflow (catastrophe-hist pipeline only on this target).
print(f"Source: {CATALOG}.{SIMULATOR_SCHEMA}.catastrophe_hist_*")
print(f"Target: {CATALOG}.lakeflow")

In [ ]:
import os
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import pipelines as p

w = WorkspaceClient()

root_abs_path = os.path.abspath("../pipelines/catastrophe_hist")
root_dbx_path = root_abs_path.replace(
    os.environ.get("DATABRICKS_WORKSPACE_ROOT", "/Workspace"),
    "/Workspace"
)

PIPELINE_NAME = f"Catastrophe Historical Aggregation ({CATALOG})"

pipeline_config = dict(
    catalog=CATALOG,
    schema="lakeflow",
    continuous=False,
    name=PIPELINE_NAME,
    serverless=True,
    configuration={
        "HIST_CATALOG": CATALOG,
        "HIST_SCHEMA": SIMULATOR_SCHEMA,
    },
    root_path=root_dbx_path,
    libraries=[p.PipelineLibrary(glob=p.PathPattern(include=f"{root_dbx_path}/**"))],
)

existing_pipelines = [
    pl for pl in w.pipelines.list_pipelines(filter=f"name LIKE '{PIPELINE_NAME}'")
    if pl.name == PIPELINE_NAME
]

if existing_pipelines:
    pipeline_id = existing_pipelines[0].pipeline_id
    w.pipelines.update(pipeline_id=pipeline_id, **pipeline_config)
    print(f"♻️ Updated existing pipeline: {pipeline_id}")
else:
    created = w.pipelines.create(**pipeline_config)
    pipeline_id = created.pipeline_id
    import sys
    sys.path.append("../utils")
    from uc_state import add
    add(CATALOG, "pipelines", created)
    print(f"✅ Created pipeline: {pipeline_id}")

In [ ]:
# Historical data is static (generated once by Catastrophe_History), so run the
# aggregation once (triggered) and wait for it to finish.
update = w.pipelines.start_update(pipeline_id=pipeline_id)
print(f"🚀 Started pipeline update: {update.update_id}")

while True:
    info = w.pipelines.get(pipeline_id=pipeline_id)
    latest = info.latest_updates[0] if info.latest_updates else None
    state_str = str(latest.state) if latest else "STARTING"
    if "COMPLETED" in state_str:
        print(f"✅ Pipeline finished: {state_str}")
        break
    if "FAILED" in state_str:
        raise RuntimeError(f"Pipeline failed: {state_str}")
    if "CANCELED" in state_str:
        raise RuntimeError(f"Pipeline canceled: {state_str}")
    print(f"  Pipeline state: {state_str}...")
    time.sleep(15)

In [ ]:
print(f"✅ Catastrophe history pipeline stage complete (pipeline_id={pipeline_id})")